In [ ]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

In [ ]:
!sh ./mario-the-explorer/setup.sh

In [ ]:
!pip install -q stable-baselines3

In [ ]:
from typing import Optional
from enum import Enum
from logging import Logger

import torch
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import KVWriter, Logger as PpoLogger

from mario_the_explorer import (MultiAttemptSuperMarioWorldEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger,
                                tile_absolute_id, TileEncoder, SuperMarioAction, SuperMarioCombo, SuperMarioDiscretizer,
                                prime_policy_for_combo, TileType)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class Direction(Enum):
    LEFT = 0
    RIGHT = 1
    UP = 2
    DOWN = 3

In [ ]:
from mario_the_explorer.environment import tiles
class TryThingsRewardModel(RewardModel):
    def __init__(self):
        self._blocks_seen = set()
        self._block_action_counts = {}
        self._combo_usage = np.zeros(11, dtype=np.float32)

    def reset(self) -> None:
        self._blocks_seen = set()
        self._block_action_counts = {}
        self._combo_usage = np.zeros(11, dtype=np.float32)

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        combo_id = SuperMarioCombo.get_combo_id_from_action(action)
        self._combo_usage[combo_id] += 1
        reward = -0.01
        for row in observation:
            for tile in row:
                tile_id = tile_absolute_id(tile)
                if tile_id not in self._blocks_seen:
                    self._blocks_seen.add(tile_id)
                    reward += 10.0
        tiles_around_mario = self._get_tiles_around_mario(observation)
        for tile_and_direction in tiles_around_mario:
            interaction_id = (combo_id, tile_and_direction[0], tile_and_direction[1])
            action_reward = 0.0
            if interaction_id not in self._block_action_counts:
                self._block_action_counts[interaction_id] = 0
            self._block_action_counts[interaction_id] += 1
            action_reward = 1.0 / (self._block_action_counts[interaction_id]**2)
            if action_reward < 0.05:
                action_reward = 0.0
            reward += action_reward
        return reward

    def _get_tiles_around_mario(self, observation: list[list[Tile]]) -> set[tuple[Direction, int]]:
        mario_coordinates = self._find_mario_coordinates(observation)
        blocks_around_mario = set()
        if not mario_coordinates:
            return blocks_around_mario
        for mario_row, mario_col in mario_coordinates:
            if mario_row > 0:
                block_above_mario = observation[mario_row - 1][mario_col]
                if block_above_mario["type"] != TileType.MARIO and block_above_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.UP.name, tile_absolute_id(block_above_mario)))
            if mario_row < len(observation) - 1:
                block_below_mario = observation[mario_row + 1][mario_col]
                if block_below_mario["type"] != TileType.MARIO and block_below_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.DOWN.name, tile_absolute_id(block_below_mario)))
            if mario_col > 0:
                block_left_of_mario = observation[mario_row][mario_col - 1]
                if block_left_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.LEFT.name, tile_absolute_id(block_left_of_mario)))
            if mario_col < len(observation[0]) - 1:
                block_right_of_mario = observation[mario_row][mario_col + 1]
                if block_right_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.RIGHT.name, tile_absolute_id(block_right_of_mario)))
        return blocks_around_mario


    def _find_mario_coordinates(self, observation: list[list[Tile]]) -> list[tuple[int, int]]:
        mario_coordinates = []
        for row_idx, row in enumerate(observation):
            for col_idx, tile in enumerate(row):
                if tile["type"] == TileType.MARIO:
                    mario_coordinates.append((row_idx, col_idx))
        return mario_coordinates

    def get_stats_vector(self):
        # Normalize: log scaling helps the network handle values from 1 to 10,000
        return np.log1p(self._combo_usage) / 10.0

    def get_potential_map(self, observation: list[list[Tile]]) -> np.ndarray:
        h, w = len(observation), len(observation[0])
        potential_map = np.zeros((h, w), dtype=np.float32)

        for r in range(h):
            for c in range(w):
                t_id = tile_absolute_id(observation[r][c])
                # Count how many (combo, direction) pairs have 0 interactions
                fresh_interactions = 0
                # You'd iterate through your SuperMarioCombo list here
                for combo in range(11):
                    for d in ["UP", "DOWN", "LEFT", "RIGHT"]:
                        if (combo, d, t_id) not in self._block_action_counts:
                            fresh_interactions += 1

                # Normalize (0.0 = fully explored, 1.0 = brand new block)
                potential_map[r, c] = fresh_interactions / 44.0
        return potential_map

In [ ]:
class MarioReshapeWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # Get the original H, W (14, 16)
        h, w = self.observation_space.shape
        # Redefine the space to include a single channel (1, 14, 16)
        self.observation_space = gym.spaces.Dict({
            "screen": gym.spaces.Box(low=0, high=255, shape=(2, 14, 16), dtype=np.float32), # IDs + Potential
            "stats": gym.spaces.Box(low=0, high=1, shape=(11,), dtype=np.float32)          # Global Combo Usage
        })

    def observation(self, obs):
        reward_model = self.env.unwrapped.reward_model
        heatmap = reward_model.get_potential_map(self.env.unwrapped.observation)
        screen = np.stack([obs, heatmap], axis=0).astype(np.float32)
        stats = reward_model.get_stats_vector()
        return {"screen": screen, "stats": stats}

In [ ]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import torch.nn as nn

class CustomMarioCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 64):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 4, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Conv2d(4, 16, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute shape by doing one forward pass
        with torch.no_grad():
            sample_tensor = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample_tensor).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU()
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.linear(self.cnn(observations))

In [ ]:
class CustomMarioDictCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Dict, features_dim: int = 128):
        # We define a temporary dim for the CNN output
        cnn_output_dim = 128
        super().__init__(observation_space, features_dim)

        n_input_channels = observation_space["screen"].shape[0] # 2 channels

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute CNN output size
        with torch.no_grad():
            sample_screen = torch.as_tensor(observation_space["screen"].sample()[None]).float()
            n_flatten = self.cnn(sample_screen).shape[1]

        self.cnn_fc = nn.Linear(n_flatten, cnn_output_dim)

        # Final Layer: CNN Features (128) + Stats Vector (11)
        self.combined_fc = nn.Sequential(
            nn.Linear(cnn_output_dim + 11, features_dim),
            nn.ReLU()
        )

    def forward(self, observations) -> torch.Tensor:
        # Process the screen
        cnn_features = self.cnn_fc(self.cnn(observations["screen"]))

        # Process the stats
        stats_features = observations["stats"]

        # Concatenate and pass through final layer
        combined = torch.cat([cnn_features, stats_features], dim=1)
        return self.combined_fc(combined)

In [ ]:
RUN_NAME = "cnn_action_rewards"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"
ATTEMPTS = 1

In [ ]:
class PpoKvWriter(KVWriter):
    def __init__(self, logger: Logger):
        self._logger = logger

    def write(self, key_values, key_excluded, step=0):
        for key, value in key_values.items():
            self._logger.info(f"Step {step} - {key}: {value}")

    def close(self):
        pass

In [ ]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
ppo_logger = PpoLogger(
    folder=None,
    output_formats=[PpoKvWriter(logger)]
)
base_env = MultiAttemptSuperMarioWorldEmulator(level = LEVEL,
                                               render_mode = "rgb_array",
                                               reward_model = TryThingsRewardModel(),
                                               attempts = ATTEMPTS,
                                               render_debug = True,
                                               render_grid = True,
                                               logger = logger)
env = SuperMarioDiscretizer(base_env)
env = MarioReshapeWrapper(env)

In [ ]:
try:
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioDictCNN,
        features_extractor_kwargs=dict(features_dim=128),
    )
    model = PPO("MultiInputPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=10000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=60000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

In [ ]:
try:
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioDictCNN,
        features_extractor_kwargs=dict(features_dim=128),
    )
    model = PPO("MultiInputPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=20000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=500000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"trained-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

In [ ]:
try:
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioDictCNN,
        features_extractor_kwargs=dict(features_dim=128),
    )
    model = PPO("MultiInputPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=20000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=5000000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"full-train-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()